# 03 - Processes and CPU work

**Time:** about 30 minutes, including the exercises.  
**You need:** basic functions, lists, loops, and the idea of threads from notebook 02.  
**Requirements:** Python 3.10 or newer. No extra packages, downloads, or GPU are needed for the examples.

By the end, you will be able to start a process, use a process pool, explain why process memory is separate, and identify when moving data can cost more than computing on it.

Run the cells from top to bottom. The default examples use at most **two worker processes** and eight tiny jobs. Predictions and exercises matter more than the timing numbers.

[Course guide](README.md) · Previous: [Threads and I/O](02_threads_and_io.ipynb) ·
Next: [Asyncio](04_asyncio.ipynb)

## 1. A process is a separate running program

A Python **process** has its own interpreter and its own ordinary Python objects. A **thread** runs inside a process and shares that process's memory.

In a usual CPython build with the GIL enabled, only one thread in an interpreter executes Python code at a time. Separate processes have separate interpreters, so CPU work written in Python can run on several cores. Optional free-threaded builds change the thread limitation; this lesson does not assume you have one. Native code in libraries can also release the GIL.

| Choice | Useful starting point | Cost to remember |
|---|---|---|
| Sequential loop | Small jobs; simplest baseline | One job at a time |
| Thread pool | Jobs that spend time waiting for I/O | Shared objects need care |
| Process pool | Substantial CPU work written in Python | Startup, extra memory, and moving inputs/results |

Processes are a tool for parallel execution, not a promise of faster code. See the [Python multiprocessing introduction](https://docs.python.org/3/library/multiprocessing.html#introduction).

In [1]:
from pathlib import Path
import inspect
import pickle
import subprocess
import sys

# Find the project folder whether Jupyter starts here or in a subfolder.
ROOT = next(
    (folder for folder in (Path.cwd(), *Path.cwd().parents)
     if (folder / "lessons" / "process_demo.py").is_file()),
    None,
)
if ROOT is None:
    raise FileNotFoundError(
        "Start Jupyter from the course folder and keep lessons/ beside the notebooks."
    )
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from lessons import process_demo


def run_demo(*arguments):
    completed = subprocess.run(
        [sys.executable, str(ROOT / "lessons" / "process_demo.py"), *arguments],
        cwd=ROOT,
        capture_output=True,
        text=True,
        check=True,
        timeout=120,
    )
    print(completed.stdout)


print("Python:", sys.version.split()[0])
print("Helper:", ROOT / "lessons" / "process_demo.py")

Python: 3.13.7
Helper: C:\Users\stefa\Desktop\workspace\parallel_processing\lessons\process_demo.py


## 2. Why this notebook runs a small Python file

**Spawn** means starting a fresh Python interpreter. That interpreter must be able to find the worker function in a Python module. Functions defined only inside a notebook cell can fail this requirement.

We put workers in `lessons/process_demo.py` and launch that file using the notebook's Python interpreter, `sys.executable`. The notebook shows the source before using it, so you can inspect the whole example.
`inspect.getsource(function)` prints the function's Python code; that printed
code is what the helper runs.

The setup cell finds the project folder by looking for the helper. `subprocess.run`
launches a separate program and waits for it. `--demo`, `--workers`, and similar
strings are command-line options the helper reads. `*arguments` passes each
option as its own list item. You can use `run_demo` without mastering this
launcher; focus first on the worker and pool code it displays.

The file ends with this **main guard**:

```python
if __name__ == "__main__":
    main()
```

This says: run `main()` when the file is launched as a program. Do not run it merely because another interpreter imports the file. Without this guard, importing the file could try to start more processes again.

We explicitly select `get_context("spawn")`, so this lesson uses the same start method on Windows, macOS, and Linux. See [Python's spawn guidance](https://docs.python.org/3/library/multiprocessing.html#the-spawn-and-forkserver-start-methods).

## 3. Start one process

**Predict first:** the parent passes `["parent"]` to a child. The child appends `"child"`. Will the parent's list change?

Read the code below. `start()` starts the child. `join()` waits for it. The
timeout and exit-code checks make a failed example visible: an **exit code**
of zero means the child ended normally; another value signals a problem.
`args=(names,)` is a tuple containing one argument. The comma makes it a tuple.

In [2]:
print(inspect.getsource(process_demo.append_in_child))
print(inspect.getsource(process_demo.memory_demo))

def append_in_child(names):
    """Change this process's copy of a regular Python list."""
    names.append("child")
    print(f"Child PID {os.getpid()}: {names}", flush=True)

def memory_demo():
    names = ["parent"]
    context = mp.get_context("spawn")
    process = context.Process(target=append_in_child, args=(names,))
    print(f"Parent PID {os.getpid()}, before: {names}", flush=True)
    process.start()
    process.join(timeout=20)  # Wait for the child to finish.
    if process.is_alive():
        process.terminate()
        process.join()
        raise TimeoutError("The small child-process demo took too long.")
    if process.exitcode != 0:
        raise RuntimeError(f"Child exited with code {process.exitcode}")
    print(f"Parent PID {os.getpid()}, after:  {names}")
    assert names == ["parent"]



In [3]:
run_demo("--demo", "memory")

Parent PID 3212, before: ['parent']
Child PID 19548: ['parent', 'child']
Parent PID 3212, after:  ['parent']



The parent still has `["parent"]`. With spawn, the child receives its own copy of the ordinary list. Its change does not update the parent's list.

The two **PIDs** are process identifiers: numbers the operating system gives running processes. Their actual values change each run.

A process does not automatically share a regular Python list. To exchange information deliberately, you can return results through a pool, or use tools such as a multiprocessing queue. Shared memory is another option, with extra coordination work.

## 4. Give every approach the same CPU work

Our job repeatedly squares an integer, takes a remainder, and adds it to a score. `** 2` means square; `% 97` means remainder after division by 97.

This is **synthetic CPU work**: a small artificial task that spends its time executing Python arithmetic. It does not decode images, read files, or simulate waiting with `sleep`.

Each input is `(job_id, iterations)`. Each output is `(job_id, score)`. The job ID lets us check which input produced which result.

In [4]:
print(inspect.getsource(process_demo.cpu_score))
print("A very small job:", process_demo.cpu_score((0, 5)))

def cpu_score(task):
    """Synthetic CPU work: repeated Python arithmetic, not image decoding."""
    job_id, iterations = task
    score = 0
    for value in range(iterations):
        score += ((value + job_id) ** 2) % 97
    return job_id, score

A very small job: (0, 30)


For this small job, the score is `0**2 + 1**2 + 2**2 + 3**2 + 4**2 = 30`. The remainder operation leaves these small values unchanged. Larger jobs keep the CPU busy by repeating the same calculation many times.

The worker is defined at the **top level of the module**: it is not inside another function. Processes need to find the function by its module and name. Its inputs and outputs must also be **picklable**, meaning Python can turn them into bytes and reconstruct them. Small tuples of integers work well here.

## 5. Compare sequential code, threads, and processes

A **pool** keeps a limited group of workers available for many jobs. We use eight jobs and two workers; that does not create eight processes.

`pool.map(function, inputs)` applies the function to each input. Its returned results follow **input order**, even when jobs finish in a different order. `list(...)` consumes those results and waits for them. Leaving the `with` block shuts down the pool.

**Predict:** with tiny jobs, which approach might pay more setup cost than it saves?

In [5]:
print(inspect.getsource(process_demo.compare_demo))

def compare_demo(iterations=120_000, jobs=8, workers=2, chunksize=1):
    tasks = [(job_id, iterations) for job_id in range(jobs)]

    start = perf_counter()
    sequential = [cpu_score(task) for task in tasks]
    sequential_seconds = perf_counter() - start

    start = perf_counter()
    with ThreadPoolExecutor(max_workers=workers) as pool:
        threaded = list(pool.map(cpu_score, tasks))
    thread_seconds = perf_counter() - start

    context = mp.get_context("spawn")
    start = perf_counter()
    with ProcessPoolExecutor(max_workers=workers, mp_context=context) as pool:
        processed = list(pool.map(cpu_score, tasks, chunksize=chunksize))
    process_seconds = perf_counter() - start

    assert sequential == threaded == processed
    print(f"Jobs: {jobs}; iterations per job: {iterations:,}; workers: {workers}")
    print(f"Process chunksize: {chunksize}; logical CPUs reported: {os.cpu_count()}")
    print(f"Sequential: {sequential_seconds:.4f} s")
    print(f"Threads:    

In [6]:
run_demo(
    "--demo", "compare",
    "--iterations", "120000",
    "--jobs", "8",
    "--workers", "2",
    "--chunksize", "1",
)

Jobs: 8; iterations per job: 120,000; workers: 2
Process chunksize: 1; logical CPUs reported: 16
Sequential: 0.0477 s
Threads:    0.0563 s
Processes:  0.1138 s
All three methods returned the same results.
First two results: [(0, 5759760), (1, 5759784)]
Pool timings include worker startup and shutdown.



The check compares **all results**, not just their count. Correct results come first.

Read the times as measurements from this run. Each pool time includes starting and stopping its workers. Small jobs often finish faster sequentially. On a GIL-enabled build, threads generally do not make this pure Python loop faster. Processes may help once there is enough computation to repay their overhead.

Other programs, CPU power settings, available cores, and the Python build can change the numbers. No cell assumes a particular method wins. The reported logical CPU count also need not equal physical cores or the CPU capacity available to this program.

For the API details, see [ProcessPoolExecutor and map](https://docs.python.org/3/library/concurrent.futures.html#processpoolexecutor).

## 6. Moving data is work too

**Serialization** means converting an object into bytes so it can be sent elsewhere. **Deserialization** rebuilds an object from those bytes. Multiprocessing commonly uses Python's `pickle` machinery for this transfer.

A file path is a small description of where data lives. Decoded image pixels can be much larger. Sending the path lets the worker read and decode the image itself, though the file access and decoding still take time.

The next cell compares serialized sizes. It is a size demonstration, not a transfer-speed benchmark. The pretend path is never opened.

In [7]:
image_path = "images/example_001.raw"
# A pretend 224 x 224 RGB image: 3 one-byte channels for every pixel.
image_pixels = bytes(224 * 224 * 3)

for label, value in [("Path", image_path), ("Decoded pixels", image_pixels)]:
    serialized = pickle.dumps(value, protocol=pickle.HIGHEST_PROTOCOL)
    print(f"{label:15} -> {len(serialized):,} serialized bytes")

# Local round trip: reconstruct a bytes object with the same contents.
assert pickle.loads(pickle.dumps(image_pixels)) == image_pixels

Path            -> 37 serialized bytes
Decoded pixels  -> 150,537 serialized bytes


Returning a large result can be expensive too. A worker that receives a short path but returns a huge array still has to send that array back.

In a real image job, consider whether each worker can return a small summary
or an output path after saving its result.

A **tensor** is an array of numbers arranged in dimensions, such as image
height, width, and channels. If training needs the full tensor, the pipeline
still needs to move that tensor; measure that cost.

A process also uses memory for its interpreter, imports, temporary objects, and results. A short input does not remove those costs. See [Python's explanation of pickle](https://docs.python.org/3/library/pickle.html).

## 7. Eight cores and millions of images

A reasonable first design is:

1. Keep a list or stream of image paths or IDs.
2. Give a limited number of jobs to a process pool.
3. Read, decode, and preprocess inside the workers.
4. Consume or save results as the jobs finish.
5. Measure the whole pipeline and adjust worker count and batch size.

Eight cores do not imply an eightfold speedup. Disk access, copying, available memory, and downstream GPU work can limit throughput. Some image libraries already use native threads or release the GIL; compare threads too when using those libraries.

Here is a deliberately rough **memory budget**, not a measurement. Suppose each of eight workers holds sixteen decoded 1024 x 1024 RGB images with one byte per channel.

In [8]:
workers = 8
images_per_worker = 16
height = width = 1024
channels = 3
bytes_per_channel = 1

bytes_per_image = height * width * channels * bytes_per_channel
active_pixel_bytes = workers * images_per_worker * bytes_per_image
print(f"One image: {bytes_per_image / 1024**2:.1f} MiB")
print(f"Pixels held by all workers: {active_pixel_bytes / 1024**2:.0f} MiB")
print("This excludes interpreters, results, queued batches, and temporary copies.")

One image: 3.0 MiB
Pixels held by all workers: 384 MiB
This excludes interpreters, results, queued batches, and temporary copies.


`MiB` means 1,048,576 bytes. This example already holds **384 MiB** of pixels. Converting those same pixels to 32-bit floats would use four times as much pixel storage. Extra copies increase it again.

More workers can increase both throughput and memory use. Measure both. Start small and raise the worker count gradually.

## 8. Pool chunks and application batches

These ideas are related, but they act at different levels.

| Setting | What changes? | What does the worker function receive? |
|---|---|---|
| `map(cpu_score, tasks, chunksize=4)` | The process pool groups several jobs for scheduling and transfer | One task per call to `cpu_score` |
| `map(cpu_score_batch, batches)` | Your code defines a batch as one input | A list of tasks per call to `cpu_score_batch` |

**Scheduling** means assigning work to available workers. A larger process
`chunksize` can reduce the time spent assigning and transferring many small
jobs. Very large chunks can leave workers idle when some groups take longer
than others. Thread pools ignore `chunksize`.

An application batch lets your function do something with a whole group,
such as one tensor operation. Our simple batch function just loops.
**Vectorized computation** means asking a library to operate on a whole array
at once; our grouping example does not do that. See the [map documentation](https://docs.python.org/3/library/concurrent.futures.html#concurrent.futures.Executor.map).

In [9]:
print(inspect.getsource(process_demo.cpu_score_batch))
print(inspect.getsource(process_demo.batch_demo))

def cpu_score_batch(batch):
    """An application batch is explicitly a list of individual tasks."""
    return [cpu_score(task) for task in batch]

def batch_demo(iterations=120_000, jobs=8, workers=2, batch_size=2):
    tasks = [(job_id, iterations) for job_id in range(jobs)]
    batches = [tasks[index:index + batch_size]
               for index in range(0, len(tasks), batch_size)]
    context = mp.get_context("spawn")
    with ProcessPoolExecutor(max_workers=workers, mp_context=context) as pool:
        nested_results = list(pool.map(cpu_score_batch, batches, chunksize=1))
    # Each worker call returns a list; combine those lists into one list.
    results = [item for batch in nested_results for item in batch]
    assert results == [cpu_score(task) for task in tasks]
    print(f"{len(tasks)} jobs grouped into {len(batches)} application batches.")
    print(f"Items per returned batch: {[len(batch) for batch in nested_results]}")
    print("Batched results match the sequential resul

In [10]:
run_demo("--demo", "batches", "--jobs", "8", "--workers", "2", "--batch-size", "2")

8 jobs grouped into 4 application batches.
Items per returned batch: [2, 2, 2, 2]
Batched results match the sequential results.



### A larger chunksize does not bound the queue

`chunksize` is not a memory limit. This lesson deliberately uses only eight jobs.

A **generator** supplies inputs as they are requested. In Python 3.10
through 3.13, `Executor.map` requests and submits all inputs immediately.
Passing it a generator therefore does not guarantee bounded memory.
Python 3.14 added a `buffersize` option to limit submitted work whose results
have not yet been returned to the caller. This lesson also runs on Python 3.10.

For millions of images, limit **work in flight**: jobs already submitted
whose results have not yet been consumed. The backpressure notebook shows why a producer must slow down when its consumer falls behind. See [map's buffering behavior](https://docs.python.org/3/library/concurrent.futures.html#concurrent.futures.Executor.map).

## 9. Practice: explain the separate lists

Before opening the answer, explain these two points in your own words:

1. Why did the child's `append` not change the parent's list?
2. Would putting a `threading.Lock` around that `append` make the processes share the list?

<details>
<summary>Worked answer</summary>

1. Spawn started a separate interpreter and reconstructed a separate list for the child. The two lists can contain the same values without being the same object.
2. No. A lock controls access; it does not turn independent objects into shared memory. Use an explicit communication mechanism, such as returning a result through a process pool, when the parent needs the child's result.

</details>

## 10. Practice: increase the useful work

Change `RUN_LARGER` to `True` to run this optional experiment. Try `600_000` iterations per job, then compare with the earlier run. Keep two workers initially.

Write down: which times changed, whether processes now helped, and one reason the result might differ on another machine. Do not keep increasing the workload just to force a particular winner.

In [11]:
RUN_LARGER = False
larger_iterations = 600_000

if RUN_LARGER:
    run_demo(
        "--demo", "compare",
        "--iterations", str(larger_iterations),
        "--workers", "2",
        "--jobs", "8",
    )
else:
    print("Optional experiment skipped. Set RUN_LARGER = True to try it.")

Optional experiment skipped. Set RUN_LARGER = True to try it.


<details>
<summary>Worked interpretation</summary>

Increasing iterations increases the arithmetic in each job while keeping the number of inputs and their size nearly unchanged. Startup and transfer costs therefore become smaller fractions of the total.

Processes may improve relative to the sequential loop, but there is no required ordering. Limited CPU availability, other programs competing for resources, and the
size of the increase all matter. For a careful benchmark, repeat measurements, compare typical times, and consider whether the real application will reuse an existing pool.

Threads on a GIL-enabled build still take turns executing this Python loop. Their behavior can differ when the worker calls native library operations that release the GIL.

</details>

## 11. Practice: distinguish chunking from batching

There are eight input jobs and two worker processes.

1. With `chunksize=4`, how many times is `cpu_score` called?
2. If you create two application batches of four jobs, how many times is `cpu_score_batch` called?
3. Does either setting by itself prevent unlimited queued work in a much larger pipeline?

<details>
<summary>Worked answer</summary>

1. Eight calls. The pool groups the scheduling of the jobs, but still calls `cpu_score` once for each job.
2. Two calls. Your batch function receives a list of four tasks on each call. Our implementation then calls `cpu_score` four times inside each batch, giving eight individual scores in total.
3. No. Group size and the limit on outstanding work are separate choices. The pipeline needs a bound on how much work or data can wait ahead of the consumer.

To see the second case, change the previous batch demo to `--batch-size 4`. You should get two returned lists containing four results each.

</details>

## 12. Practice: explain an MLE design

Imagine you have eight CPU cores and millions of images. Decoding and Python preprocessing are keeping the GPU waiting. Give a short plan that mentions workers, data transfer, memory, and measurement.

<details>
<summary>One worked answer</summary>

I would measure where time is spent first. I would try a small process pool that receives image paths and reads and preprocesses inside each worker. I would compare the output with a sequential baseline and then compare end-to-end throughput. I would test two workers, then increase gradually if CPU capacity and memory allow it.

I would also compare threads if the image library releases the GIL. I would avoid returning unnecessary large intermediate objects. I would limit queued batches so CPU producers cannot exhaust memory while the GPU is busy. If training with PyTorch, I would start with its DataLoader worker and prefetch controls instead of building all of this myself.

Eight cores are a resource to measure against, not a required worker count or a guarantee of eight times the speed.

</details>

## When a process example fails

| Symptom | First thing to check |
|---|---|
| Cannot pickle a function; cannot find a worker | Is the worker defined at module level in a `.py` file, rather than a cell, lambda, or nested function? |
| Error about starting a process during startup | Is the process-launching code protected by the main guard? |
| Pool is slower than the loop | Are jobs too small to repay startup and data transfer? |
| Memory climbs as inputs grow | Are large objects being copied or too many jobs/results being queued? |

`run_demo` uses `check=True` to surface script failures and a 120-second timeout to catch a stuck demonstration. It is a teaching launcher, not a general process supervisor. After a `CalledProcessError`, its exception's `stderr` contains the helper's error output.

You can also run the helper directly from the project folder:

```text
python lessons/process_demo.py --demo memory
python lessons/process_demo.py --demo compare --workers 2 --iterations 600000
python lessons/process_demo.py --demo batches --batch-size 4
```

Next: [04 · Asyncio](04_asyncio.ipynb), followed by backpressure and PyTorch data loading. You now have the process model needed to understand worker-based input pipelines.